In [ ]:
!pip install sentence-transformers nltk dspy fastembed pip-system-certs faiss-cpu gradio openpyxl --upgrade

In [ ]:
import os

# Define the path to the extracted folder
extracted_folder_path = "C:/TEMPLATES"
print(f"Extracted folder path defined as: {extracted_folder_path}")

# Initialize an empty list to store the paths of the found XML files
xml_files_recursive = []

try:
    # Use os.walk() to traverse through all directories and subdirectories
    for root, dirs, files in os.walk(extracted_folder_path):
        # Iterate through the files in the current directory
        for file in files:
            # Check if the file has a .xml extension (case-insensitive)
            if file.lower().endswith(".xml"):
                # Construct the full path to the file
                full_file_path = os.path.join(root, file)
                # Append the full path to the list
                xml_files_recursive.append(full_file_path)

    # Print the list of found XML files
    print(f"Found XML files recursively in '{extracted_folder_path}': {xml_files_recursive}")

except FileNotFoundError:
    print(f"Error: The specified extracted folder path '{extracted_folder_path}' was not found.")
except Exception as e:
    print(f"An unexpected error occurred during recursive file listing: {e}")

In [ ]:
import os

excel_file_name = "TEMPLATES_MAPPING.xlsx"
found_excel_path = None

for root, dirs, files in os.walk(extracted_folder_path):
    if excel_file_name in files:
        found_excel_path = os.path.join(root, excel_file_name)
        break

if found_excel_path:
    print(f"Found Excel file at: {found_excel_path}")
else:
    print(f"Excel file '{excel_file_name}' not found in '{extracted_folder_path}' or its subdirectories.")

In [ ]:
import pandas as pd

excel_file_path = "C:/TEMPLATES/TEMPLATES_MAPPING.xlsx"
df_mapping = pd.read_excel(excel_file_path)
display(df_mapping.head())

In [ ]:
for index, row in df_mapping.iterrows():
    query_name = row["NAME"]
    test_name = row["VALUE"]
    print(f"Query Name: {query_name}, Test Name: {test_name}")

In [ ]:
import xml.etree.ElementTree as ET
import pandas as pd
import io

# Initialize a list to store the data for the new DataFrame
data_for_df = []

# Create a dictionary to map test names from the Excel to their found locations and snippets
test_name_occurrences = {}

for xml_file_path in xml_files_recursive:
    try:
        tree = ET.parse(xml_file_path)
        root = tree.getroot()

        # Iterate through all <test> elements in the current XML file
        for test_element in root.findall(".//test"):
            test_name_in_xml = test_element.get("name")

            # Check if this test name exists in our df_mapping VALUE column
            matching_rows = df_mapping[df_mapping['VALUE'] == test_name_in_xml]

            if not matching_rows.empty:
                # If the test name from XML matches a VALUE in df_mapping, process it
                xml_snippet_element = ET.tostring(test_element, encoding='unicode', method='xml')

                for index, row in matching_rows.iterrows():
                    query_name = row["NAME"]
                    test_name_from_excel = row["VALUE"] # This should be the same as test_name_in_xml

                    # Append the data to the list
                    data_for_df.append({
                        "NAME": query_name,
                        "VALUE": test_name_from_excel,
                        "XML_Snippet": xml_snippet_element,
                        "XML_File": xml_file_path # Add file path for reference
                    })

                    # Optionally, store occurrences in a dictionary for verification or further use
                    if test_name_from_excel not in test_name_occurrences:
                        test_name_occurrences[test_name_from_excel] = []
                    test_name_occurrences[test_name_from_excel].append({
                        "XML_File": xml_file_path,
                        "XML_Snippet": xml_snippet_element
                    })


    except FileNotFoundError:
        print(f"Error: XML file not found at '{xml_file_path}'.")
    except ET.ParseError as e:
        print(f"Error parsing XML file '{xml_file_path}': {e}")
    except Exception as e:
        print(f"An unexpected error occurred while processing '{xml_file_path}': {e}")


# Create the DataFrame from the collected data
df_results = pd.DataFrame(data_for_df)

# Print the total number of test elements found
print(f"\nTotal number of matching test elements found based on Excel mapping: {len(data_for_df)}")

# Optional: Verification step - check which test names from Excel were not found in any XML
excel_test_names = df_mapping['VALUE'].unique()
found_test_names = test_name_occurrences.keys()
not_found_test_names = [name for name in excel_test_names if name not in found_test_names]

if not_found_test_names:
    print(f"\nWarning: The following test names from the Excel mapping were not found in any XML file: {not_found_test_names}")
else:
    print("\nAll test names from the Excel mapping were found in the XML files.")


print("\nSearch complete. Results DataFrame created:")
display(df_results.head())

In [ ]:
from sentence_transformers import SentenceTransformer
# Instantiate the SentenceTransformer model
model = SentenceTransformer("C:/all-MiniLM-L6-v2")
print("SentenceTransformer model loaded successfully.")

In [ ]:
# Generate embeddings for the NAME column
# Use the 'NAME' column from the df_mapping DataFrame
embeddings = model.encode(df_mapping['NAME'].tolist())

print(f"Generated embeddings for {len(df_mapping)} NAMES.")
print(f"Shape of embeddings: {embeddings.shape}")

In [ ]:
import faiss
import numpy as np

# Convert embeddings to a numpy array with float32 datatype, as required by FAISS
embeddings = np.array(embeddings).astype('float32')

# Get the dimension of the embeddings
dimension = embeddings.shape[1]

# Create a FAISS index (using IndexFlatL2 for simplicity)
index = faiss.IndexFlatL2(dimension)

# Add the embeddings to the index
index.add(embeddings)

print(f"FAISS index created with {index.ntotal} embeddings.")

In [ ]:
def retrieve_test_name_and_snippet(query, name_index, snippet_df, model, df_mapping, k_name=1, k_snippet=1):
    """
    Retrieves the top-k relevant test names (VALUE) based on a query by searching NAME embeddings,
    and then retrieves the corresponding XML snippets from the snippets DataFrame.

    Args:
        query (str): The user query.
        name_index (faiss.Index): The FAISS index containing the NAME embeddings.
        snippet_df (pd.DataFrame): DataFrame containing the XML snippets (df_results).
        model (SentenceTransformer): The sentence transformer model for generating embeddings.
        df_mapping (pd.DataFrame): DataFrame containing the NAME and VALUE mapping.
        k_name (int): The number of top names to retrieve from the name index.
        k_snippet (int): The number of top snippets to retrieve for each matched name (useful if a VALUE appears multiple times).

    Returns:
        list: A list of dictionaries containing the retrieved NAME, VALUE, distance (from name search), and a list of associated XML snippets.
    """
    # Step 1: Retrieve top-k relevant test names (VALUE) based on query by searching NAME embeddings
    query_embedding = model.encode([query]).astype('float32')
    name_distances, name_indices = name_index.search(query_embedding, k_name)

    retrieved_info_with_snippets = []

    for i in range(k_name):
        # Get the index in the original df_mapping DataFrame
        original_df_index = name_indices[0][i]
        # Get the distance from the name search
        name_distance = name_distances[0][i]

        # Get the NAME and VALUE from the df_mapping DataFrame
        matched_name = df_mapping.iloc[original_df_index]["NAME"]
        retrieved_value = df_mapping.iloc[original_df_index]["VALUE"]

        # Step 2: Retrieve the corresponding XML snippets from the snippets DataFrame (df_results)
        # Find rows in snippet_df where the 'VALUE' matches the retrieved_value
        matching_snippets_df = snippet_df[snippet_df['VALUE'] == retrieved_value]

        associated_snippets = []
        if not matching_snippets_df.empty:
            # If there are multiple snippets for the same VALUE, take the top k_snippet
            # We can potentially add a similarity search here within the matching snippets
            # based on the original query, but for simplicity, let's just take the first k_snippet for now
            for j in range(min(k_snippet, len(matching_snippets_df))):
                 snippet_row = matching_snippets_df.iloc[j]
                 associated_snippets.append({
                     "XML_File": snippet_row["XML_File"],
                     "XML_Snippet": snippet_row["XML_Snippet"]
                 })
        else:
            associated_snippets.append({"XML_File": "N/A", "XML_Snippet": "No snippet found for this VALUE in df_results."})


        retrieved_info_with_snippets.append({
            "Query_Name_Matched": matched_name,
            "Retrieved_Test_Name": retrieved_value,
            "Name_Search_Distance": name_distance, # Add Name_Search_Distance back
            "Associated_Snippets": associated_snippets
        })

    return retrieved_info_with_snippets

# Example usage:
user_query = "how to change a node"
retrieved_results = retrieve_test_name_and_snippet(user_query, index, df_results, model, df_mapping, k_name=1, k_snippet=2)

print(f"Retrieval results for the query: '{user_query}'")
for result in retrieved_results:
    print(f"\nMatched Query Name: {result['Query_Name_Matched']}")
    print(f"Retrieved Test Name: {result['Retrieved_Test_Name']}")
    print(f"Name Search Distance: {result['Name_Search_Distance']:.4f}")
    print("Associated Snippets:")
    for i, snippet in enumerate(result['Associated_Snippets']):
        print(f"  Snippet {i+1}:")
        print(f"    XML File: {snippet['XML_File']}")
        print(f"    Snippet Content:\n{snippet['XML_Snippet']}")

In [ ]:
import dspy
from dspy import Predict
from dspy import ChainOfThought
import re # Import regular expression module
from collections import Counter
import nltk
from nltk.corpus import stopwords

# Download necessary NLTK data (if not already downloaded)
try:
    stopwords = set(stopwords.words('english'))
except LookupError:
    nltk.download('stopwords')
    stopwords = set(stopwords.words('english'))


# Initialize the language model as specified by the user
# Using the API key provided previously
llm = dspy.LM('<YOUR LM MODEL>', api_key='<YOUR GROQ API KEY>') # Keeping Groq model initialization
dspy.configure(lm=llm)

# Define a custom retrieval module (keeping as is)
class CustomRetriever(dspy.Retrieve):
    def __init__(self, k=5): # Increase k to retrieve more potential names initially
        super().__init__()
        self.k = k

    def forward(self, query):
        # Use the retrieve_test_name_and_snippet function to get the relevant snippets
        # The name_index, snippet_df, model, and df_mapping should be accessible in the environment
        # Retrieve more potential matches for vague queries (e.g., k_name=self.k)
        retrieved_results = retrieve_test_name_and_snippet(query, index, df_results, model, df_mapping, k_name=self.k, k_snippet=1) # Changed k_snippet to 1 as we are focusing on names first

        # Extract the XML snippets from the retrieved results to use as passages
        passages = []
        potential_names = []
        for result in retrieved_results:
             potential_names.append(result['Query_Name_Matched'])
             for snippet_info in result['Associated_Snippets']:
                  # You can choose how to format the snippet as a passage.
                  # Here, we include the file name and the snippet content.
                  passage = f"Source File: {snippet_info['XML_File']}\nXML Snippet:\n{snippet_info['XML_Snippet']}"
                  passages.append(passage)

        # print(f"CustomRetriever Output - Passages Count: {len(passages)}")
        # print(f"CustomRetriever Output - Potential Names: {potential_names}")
        # print(f"CustomRetriever Output - Retrieved Details (first): {retrieved_results[0] if retrieved_results else 'None'}")


        return dspy.Prediction(passages=passages, potential_names=potential_names, retrieved_details=retrieved_results) # Return potential names and details


# Set up the retrieval module using the custom class
retriever = CustomRetriever(k=5)


# Define the RAG module with customized forward function
class RAG(dspy.Module):
    def __init__(self, num_passages=3):
        super().__init__()
        self.retrieve = retriever
        self.generate_answer = ChainOfThought("context, query -> answer")

    def forward(self, query, context=None):
        query_lower = query.lower()

        # print(f"\nRAG Model - Processing Query: '{query}'")

        # Handle specific broad queries
        if "show me all snippets" in query_lower or "all data" in query_lower:
            all_snippets = df_results.apply(
                lambda row: f"Source File: {row['XML_File']}\nXML Snippet:\n{row['XML_Snippet']}",
                axis=1
            ).tolist()
            context = "\n\n---\n\n".join(all_snippets)
            llm_query = f"Here is all the available XML content from the folder:\n\n{context}\n\nPlease provide a summary or list of the main topics/snippets available. Do NOT include the XML snippets in your answer, only provide a summary."
            answer = self.generate_answer(context=context, query=llm_query).answer
            # print(f"RAG Model - Broad Query Handled. Context Populated.")
            # print(f"RAG Model - Answer (Broad Query): {answer[:100]}...") # Print first 100 chars of answer
            return dspy.Prediction(context=context, answer=answer, messages=[{"role": "user", "content": query}])

        # Retrieve relevant snippets and potential names
        retrieval_output = self.retrieve(query=query)
        context = retrieval_output.passages
        potential_names = retrieval_output.potential_names
        retrieved_details = retrieval_output.retrieved_details

        # print(f"RAG Model - Retrieval Output - Passages Count: {len(context)}")
        # print(f"RAG Model - Retrieval Output - Potential Names: {potential_names}")

        # Check for vagueness based on the number of distinct potential names and distance
        # Added a check for a very low distance in the top result to indicate a clear match
        is_vague = False
        if retrieved_details:
            # Check if the top result has a high distance or if there are multiple distinct names
            # with relatively low distances (indicating ambiguity)
            top_distance = retrieved_details[0]['Name_Search_Distance']
            distinct_potential_names = list(set(potential_names))

            # Refined vagueness check:
            # It's vague if the top distance is above a low threshold AND
            # either there are multiple distinct names OR the top distance is above a higher threshold.
            # You might need to tune these thresholds (0.1 and 0.2 are examples)
            if top_distance > 0.1 and (len(distinct_potential_names) > 1 or top_distance > 0.2):
                 is_vague = True
            # print(f"RAG Model - Vagueness Check - Top Distance: {top_distance:.4f}, Distinct Names Count: {len(distinct_potential_names)}, Is Vague: {is_vague}")


        if is_vague:
            # Instead of keywords, find and list matching NAMEs from df_mapping
            matching_names = []
            query_words = query_lower.split()
            # Filter df_mapping to find NAMES that contain any of the query words (case-insensitive)
            # This is a simple keyword-based matching for vague queries
            for index, row in df_mapping.iterrows():
                 name = row["NAME"].lower()
                 if any(word in name for word in query_words):
                      matching_names.append(row["NAME"]) # Append the original NAME

            # Remove duplicates and limit the list size if necessary
            matching_names = list(set(matching_names))
            # You might want to sort these by relevance or alphabetically

            if matching_names:
                 name_list_str = ", ".join(matching_names)
                 answer = f"Your query '{query}' is a bit vague. I found potential matches for the following names in our mapping: {name_list_str}. Could you please specify which one you are interested in?"
            else:
                 # Fallback if no relevant names are found in the mapping for the vague query
                 answer = f"Your query '{query}' is a bit vague and I couldn't find specific matches in our mapping. Could you please try rephrasing your query?"

            # print(f"RAG Model - Vague Query Handled. Answer: {answer}")
            return dspy.Prediction(context=[], answer=answer, messages=[{"role": "user", "content": query}])


        messages = [
            {"role": "system", "content": """
You are an XML assistant chatbot. You specialize in helping users with XML-test cases retreival tasks and extracting information of the test case acquired using XPath, based on the provided XML content from the folder.

You respond naturally to greetings like 'hi' or 'hello' with a warm welcome and a reminder of your XML expertise and focus on the provided XML data.

If a user asks something unrelated to XML or the content of the provided XML files (e.g., general knowledge, politics, celebrities, trivia), you do not answer. Instead, you gently redirect the user by saying something like:
'I'm here to help with XML-related questions based on the provided XML data. Could you ask something about the XML content?'

You do not provide general information unless it directly relates to XML or data processing *and* can be informed by the structure or content of the provided XML data.

Stay focused, helpful, and domain-specific at all times, emphasizing your role in assisting with the *provided XML content*.
"""},
            {"role": "user", "content": query}
        ]

        # If context is found (meaning explicit match was successful and not vague)
        if context:
             # print(f"RAG Model - Generating Answer with Context (Context Count: {len(context)})")
             llm_query = f"Based *only* on the provided XML content from the folder: '{' '.join(context)}', answer the user's query: '{query}'. Do NOT include the XML snippets themselves in your answer, only provide a textual response based on the information within the snippets. If the provided XML content does not contain the information needed to answer, state that the information is not found in the provided XML data."
             answer = self.generate_answer(context=context, query=llm_query).answer
             # print(f"RAG Model - Answer (Context Found): {answer[:100]}...") # Print first 100 chars of answer
        else:
             # If no relevant context is found (meaning no explicit match was found, it wasn't a broad query, and wasn't flagged as vague)
             # print(f"RAG Model - Generating Answer without Relevant Context")
             llm_query = f"The user asked '{query}'. No relevant XML content from the folder was found that explicitly matches this query. As a chatbot focused on providing information *from the XML files in the folder*, acknowledge that you didn't find relevant XML for this specific query and provide a concise, general response if possible, while reinforcing your primary focus on answering questions based on the provided XML content."
             answer = self.generate_answer(context=[], query=llm_query).answer
             # print(f"RAG Model - Answer (No Context): {answer[:100]}...") # Print first 100 chars of answer


        return dspy.Prediction(context=context, answer=answer, messages = messages)

# Instantiate the RAG module
rag_model = RAG()

print("dspy modules updated to prevent LLM from including XML snippets in its answer.")

In [ ]:
import gradio as gr
import xml.etree.ElementTree as ET
import numpy as np
import dspy
from dspy import Predict
from dspy import ChainOfThought
import re
from collections import Counter
import nltk
from nltk.corpus import stopwords

# Assuming the following are defined in previous cells and are accessible:
# model = SentenceTransformer("BAAI/bge-large-en-v1.5")
# index = faiss.IndexFlatL2(dimension)
# df_results = pd.DataFrame(...)
# df_mapping = pd.DataFrame(...)
# rag_model = RAG() # The dspy RAG model

# Download necessary NLTK data (if not already downloaded)
try:
    stopwords = set(stopwords.words('english'))
except LookupError:
    nltk.download('stopwords')
    stopwords = set(stopwords.words('english'))


def format_rag_output_for_chat(rag_output):
    """
    Formats the output of the RAG model into a single string suitable for Gradio ChatInterface.

    Args:
        rag_output (dspy.Prediction): The output object from the rag_model.

    Returns:
        str: A formatted string combining the answer and relevant snippets.
    """


    # Check if rag_output is a dspy.Prediction object before accessing attributes
    if not isinstance(rag_output, dspy.Prediction):
        return f"Error: Unexpected output format from RAG model: {type(rag_output)}"

    answer = rag_output.answer
    context = rag_output.context

    response_text_content = f"Chatbot Response:\n{answer}\n\n"

    # Check if context exists and is not empty
    if context:
        response_text_content += "--- Retrieved XML Snippets ---\n\n"
        if isinstance(context, list) and len(context) > 0:
            # Handle list of snippets, display only the top 1
            snippet_text = context[0] # Get the first snippet
            response_text_content += f"Snippet :\n"
            # Extract only the XML snippet content by splitting the passage
            # The passage is formatted as "Source File: ...\nXML Snippet:\n..."
            snippet_parts = snippet_text.split("XML Snippet:\n", 1)
            if len(snippet_parts) > 1:
                xml_snippet_content = snippet_parts[1]
            else:
                # Fallback if the format is unexpected
                xml_snippet_content = snippet_text
            response_text_content += f"```xml\n{xml_snippet_content}\n```\n\n" # Use markdown code block
        elif isinstance(context, str):
            # Handle single string context (e.g., for "show me all data")
            # Assuming the single string context is already well-formatted or we want to present it as one block
            # If it's a concatenation with separators, we might want to keep them
            response_text_content += f"```xml\n{context}\n```\n\n" # Present as one markdown code block
        else:
             # Handle unexpected context types
             response_text_content += f"Warning: Unexpected context type: {type(context)}\n\n"


        response_text_content += "-------------------------"

    return response_text_content


def chatbot_response(message, history):
    """
    Generates a response from the RAG model based on user input and updates history for ChatInterface.
    Handles comma-separated queries by processing each part as a separate turn in the returned history.
    """
    if not message:
        # Return current history if message is empty
        return history

    # Convert history to the expected list of message dictionaries format if it's not already
    formatted_history = []
    for chat_turn in history:
        if isinstance(chat_turn, list) and len(chat_turn) == 2:
            # Assuming the list format is [user_message, bot_message]
            formatted_history.append({'role': 'user', 'content': chat_turn[0]})
            formatted_history.append({'role': 'assistant', 'content': chat_turn[1]})
        elif isinstance(chat_turn, dict) and 'role' in chat_turn and 'content' in chat_turn:
            # Already in the expected dictionary format
            formatted_history.append(chat_turn)
        # Add other potential formats if necessary, or just skip/log unexpected ones
        else:
             print(f"Warning: Skipping unexpected history format: {chat_turn}")


    try:
        queries_to_process = [message.strip()] # Start with the original message

        # Check if the original message contains commas and is not an exact match in df_mapping
        if ',' in message:
            exact_match_found = False
            # Check for exact match of the full query in df_mapping['NAME'].str.lower()
            if not df_mapping[df_mapping['NAME'].str.lower() == message.strip().lower()].empty:
                exact_match_found = True

            if not exact_match_found:
                # If no exact match for the full string and contains commas, split into individual queries
                queries_to_process = [q.strip() for q in message.split(',') if q.strip()] # Split and remove empty strings

        # Process each query separately and build the new history
        new_history_entries = []
        for query in queries_to_process:
            # Call the rag_model with the individual query
            rag_output = rag_model(query=query)

            # Use the adapter function to format the output for the chat interface
            formatted_response_text = format_rag_output_for_chat(rag_output)

            # Create a new user-assistant turn for this query
            new_history_entries.append({'role': 'user', 'content': query})
            new_history_entries.append({'role': 'assistant', 'content': formatted_response_text})

        # Return the original history combined with the new entries for this input
        # This will replace the entire history in the ChatInterface
        return new_history_entries

    except Exception as e:
        # Catch any exceptions and append an informative error message to history
        error_message = f"An error occurred during processing: {type(e).__name__} - {e}"
        # Append the error message as the chatbot's response for this turn
        # Ensure the error message is clearly visible
        formatted_error_response = f"An error occurred:\n{error_message}"
        # Append the original user message and the error as a single turn at the end
        return formatted_history + [{'role': 'user', 'content': message}, {'role': 'assistant', 'content': formatted_error_response}]


# Create the Gradio ChatInterface
# Explicitly set type='messages' for better compatibility
iface = gr.ChatInterface(
    fn=chatbot_response,
    title="XML Chatbot with Snippets",
    description="Ask questions about the content of your XML files.",
    chatbot=gr.Chatbot(label="Chat History", type="messages") # Set type to messages
)

# Launch the interface
iface.launch()